# Domain Intelligence

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mamoo85/open-data-actor-notebooks/blob/main/notebooks/domain-intelligence-lookup.ipynb)

**What:** Bulk domain lookup verified by 94 passing tests: RDAP registration (registrar, created, expires, age), DNS over HTTPS (A, AAAA, MX, NS, TXT with SPF and DMARC parsed), subdomains from certificate transparency, SSL expiry and HTTP security headers. $0.002 per domain.

**Why:** run the [`domain-intelligence-lookup`](https://apify.com/transparent_meteorite/domain-intelligence-lookup) Apify actor from Python and analyse the result as a pandas DataFrame, with no scraping code to maintain.

> **Disclosure:** I am the author of this actor. It is pay-per-result; the free plan credit on a new Apify account covers a trial run like this one (this notebook uses a very small sample input).

## 1. Setup

In [1]:
!pip install -q apify-client pandas

import os, getpass
try:
    from google.colab import userdata
    APIFY_TOKEN = userdata.get('APIFY_TOKEN')          # Colab: add it under the key icon (Secrets)
except Exception:
    APIFY_TOKEN = os.environ.get('APIFY_TOKEN') or getpass.getpass('Apify API token: ')
assert APIFY_TOKEN, 'An Apify API token is required (a free account is enough).'
print('Token loaded.')

Token loaded.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Input
A deliberately small sample so the run is cheap and quick. See the [actor page](https://apify.com/transparent_meteorite/domain-intelligence-lookup) for every option.

In [2]:
run_input = {   'domains': ['example.com', 'apify.com', 'github.com'],
    'maxDomains': 3,
    'includeSubdomains': True,
    'maxSubdomains': 50,
    'checkSsl': True,
    'checkHeaders': True}

## 3. Run the actor and read the dataset

In [3]:
from apify_client import ApifyClient

client = ApifyClient(APIFY_TOKEN)
run = client.actor('transparent_meteorite/domain-intelligence-lookup').call(run_input=run_input)
# apify-client v2 returns a dict, v3 a Run object; handle both
get = (lambda k, a: run[k]) if isinstance(run, dict) else (lambda k, a: getattr(run, a))
items = list(client.dataset(get('defaultDatasetId', 'default_dataset_id')).iterate_items())
print('Run status:', get('status', 'status'), '| items:', len(items))

[apify.domain-intelligence-lookup runId:FWCComhYNxPRF3XzL] -> Status: RUNNING, Message: 


[apify.domain-intelligence-lookup runId:FWCComhYNxPRF3XzL] -> 2026-10-13T22:17:44.158Z ACTOR: Pulling container image of build mYL2RqJyZTjsrjz7L from registry.


[apify.domain-intelligence-lookup runId:FWCComhYNxPRF3XzL] -> 2026-10-13T22:17:44.160Z ACTOR: Creating container.


[apify.domain-intelligence-lookup runId:FWCComhYNxPRF3XzL] -> 2026-10-13T22:17:44.237Z ACTOR: Starting container.


[apify.domain-intelligence-lookup runId:FWCComhYNxPRF3XzL] -> 2026-10-13T22:17:44.238Z ACTOR: Running under "LIMITED_PERMISSIONS".


...

[apify.domain-intelligence-lookup runId:FWCComhYNxPRF3XzL] -> 2026-10-13T22:17:54.154Z Done: {"requested":3,"looked":3,"charged":3,"skippedInvalid":0,"duplicates":0,"chargeLimitReached":false,"warnings":[]}


[apify.domain-intelligence-lookup runId:FWCComhYNxPRF3XzL] -> 2026-10-13T22:17:54.155Z INFO  [Status message]: Done: 3 items


[apify.domain-intelligence-lookup runId:FWCComhYNxPRF3XzL] -> 2026-10-13T22:17:54.201Z Thanks for using Domain Intelligence Lookup. Found a bug or need a field? Open an issue on the Issues tab (https://console.apify.com/actors/RpmCtUq0qJd5gmekn/issues) with the run URL and we will reply there.


[apify.domain-intelligence-lookup runId:FWCComhYNxPRF3XzL] -> Status: SUCCEEDED, Message: Done: 3 items


Run status: SUCCEEDED | items: 3


## 4. Tabulate

In [4]:
import pandas as pd
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 200)

df = pd.DataFrame(items)

# Privacy: remove contact / personal-data columns before anything is displayed
DROP = ["email", "phone", "author", "registeredagent", "contractor", "officer", "linkedin", "facebook", "instagram", "youtube", "tiktok", "socials", "twitter"]
KEEP = ["emailCount", "phoneCount", "emailProvider"]
df = df[[c for c in df.columns if c in KEEP or not any(p in c.lower() for p in DROP)]]
SHOW = [c for c in ["domain", "registered", "registrar", "ageDays", "hasSpf", "hasDmarc", "dmarcPolicy", "sslIssuer", "sslDaysLeft", "securityHeadersScore", "subdomainCount"] if c in df.columns]
df[SHOW]

,domain,registered,registrar,ageDays,hasSpf,hasDmarc,dmarcPolicy,sslIssuer,sslDaysLeft,securityHeadersScore,subdomainCount
0,example.com,True,RESERVED-Internet Assigned Numbers Authority,11383,True,True,reject,SSL Corporation,73,0,6
1,apify.com,True,"Amazon Registrar, Inc.",6342,True,True,reject,Amazon,95,4,20
2,github.com,True,MarkMonitor Inc.,6944,True,True,quarantine,Sectigo Limited,47,5,42


## 5. A quick analysis

In [5]:
for c in ['ageDays','sslDaysLeft','securityHeadersScore','subdomainCount']:
    df[c]=pd.to_numeric(df[c],errors='coerce')
print(df[['domain','ageDays','sslDaysLeft','securityHeadersScore']].sort_values('ageDays',ascending=False).to_string(index=False))
print('DMARC policies:',df['dmarcPolicy'].value_counts(dropna=False).to_dict())

     domain  ageDays  sslDaysLeft  securityHeadersScore
example.com    11383           73                     0
 github.com     6944           47                     5
  apify.com     6342           95                     4
DMARC policies: {'reject': 2, 'quarantine': 1}


## Next steps

- Raise the limits in the input cell to run a larger sample.
- Schedule the actor or call it from your own pipeline: [domain-intelligence-lookup on the Apify Store](https://apify.com/transparent_meteorite/domain-intelligence-lookup).
- Check the Store page for the current pay-per-result price before scaling up.

Generated by `tools/generate.py`.